In [ ]:
from collections import Counter
from math import floor

import stim
from IPython.core.display import Markdown

from library.qubit_allocation import QubitAllocation
from library.surface_code_patch import SurfaceCodePatch

In [ ]:
def rewrite_file_with_polygons(filename: str, instructions: Counter[str], *surfaces: SurfaceCodePatch):
    # Insert all the polygons into the Stim file for readability.
    with open(filename, "r", encoding="utf-8") as file:
        lines = file.readlines()

        inserted = 0

        for surface in surfaces:
            for polygon in surface.get_polygons():
                lines.insert(instructions['metadata'] + inserted, polygon)
                inserted += 1

    with open(filename, "w", encoding="utf-8") as file:
        file.writelines(lines)
        print(f"Generated circuit : {filename}")

In [ ]:
DISTANCE = 3

grid = QubitAllocation(dimensions = (7, 3))

circuit = stim.Circuit()
instructions = Counter()
grid.append_metadata(circuit)
instructions['metadata'] = len(circuit)

surfaceS = SurfaceCodePatch(distance = DISTANCE, allocation = grid, anchor = (1,1))
surfaceT = SurfaceCodePatch(distance = DISTANCE, allocation = grid, anchor = (5,1))

for round in range(3):
    for moment in surfaceS.moments:
        surfaceS.append_syndrome_slice(circuit, moment, preparation = (round==0))
        surfaceT.append_syndrome_slice(circuit, moment, preparation = (round==0))
instructions['syndrome'] = len(circuit)

print(circuit.to_crumble_url())
display(Markdown(f"[Open in Crumble]({circuit.to_crumble_url()})"))

In [ ]:
circuit.to_file("logical-teleportation.stim")
rewrite_file_with_polygons("logical-teleportation.stim", instructions, surfaceS, surfaceT)